In [1]:
%matplotlib inline
%load_ext autoreload
%autoreload 2

In [2]:
import os
import torch
from torch.utils.data import DataLoader
import random

from data import load_coco_data, CocoDataset, decode_captions
from Transformer import TransformerCaptioner, TransformerConfig, TransformerCaptionerTrainer

In [3]:
random.seed(0)
torch.manual_seed(0)

BATCH_SIZE=25
NUM_WORKERS=1

model_save_root = "../../Models/Transformer"
os.makedirs(model_save_root,exist_ok=True)

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.mps.is_available() else "cpu"

In [4]:
# COCO captioning

data_root = '../../Data/COCO_captioning'

zip_path = os.path.join(data_root, "coco_captioning.zip")

if os.path.isfile(zip_path):
    print("COCO data exists!")
else:
    print("downloading COCO dataset")
    !wget https://cs231n.stanford.edu/coco_captioning.zip -P {data_root}
    !unzip -d {data_root} {zip_path}

COCO data exists!


In [5]:
overfit_data = load_coco_data(os.path.join(data_root,'coco_captioning'), max_train=50, pca_features=True)
word_to_idx = overfit_data['word_to_idx']
feature_dim = overfit_data['train_features'].shape[1]

train_dataset = CocoDataset(overfit_data, split='train')
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)

# Transformer

In [6]:
config=TransformerConfig(
    img_feat_dim=overfit_data["train_features"].shape[1],
    max_length=max(30, overfit_data["train_captions"].shape[1]),
)

model = TransformerCaptioner(
        word_to_idx=overfit_data["word_to_idx"],
        config=config
    )

optimizer = torch.optim.Adam(model.parameters(), lr=5e-3)

trainer=TransformerCaptionerTrainer(
    model=model, 
    optimizer=optimizer,
    device=DEVICE
)

trainer.train(
    train_dataloader=train_loader,
    save_path=os.path.join(model_save_root,'Transformer.pth'),
    num_epochs = 50,
    log_interval = 10,
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /home/chenyifeng/.netrc.
wandb: Currently logged in as: 598061731 (my59) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING The get_url method is deprecated and will be removed in a future release. Please use `run.url` instead.


✅ WandB initialized. View at: https://wandb.ai/my59/Transformer/runs/k1x9b98s
Iter: 0, Loss: 7.067
Iter: 10, Loss: 1.954
Iter: 20, Loss: 0.7742
Iter: 30, Loss: 0.2785
Iter: 40, Loss: 0.1912
Iter: 50, Loss: 0.1995
Iter: 60, Loss: 0.183
Iter: 70, Loss: 0.1527
Iter: 80, Loss: 0.1615
Iter: 90, Loss: 0.1601


train/iteration,▁▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▃▃▄▄▄▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇█
train/loss,█▅▄▄▄▃▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/iteration,99
train/loss,0.18248


✅ WandB run finished.


In [7]:
model = TransformerCaptioner(
        word_to_idx=overfit_data["word_to_idx"],
        config=config
    ).to(DEVICE)

state_dict = torch.load(os.path.join(model_save_root, 'Transformer.pth'), map_location=DEVICE)
model.load_state_dict(state_dict)
model.eval()


idx_to_word = overfit_data['idx_to_word']

for split in ["train", "val"]:
    sample_features = overfit_data[f'{split}_features'][0:2]         # (2, img_feat_dim)
    sample_captions_true = overfit_data[f'{split}_captions'][0:2]

    features_tensor = torch.tensor(sample_features, dtype=torch.float32).to(DEVICE)
    pred_captions = model.sample(features_tensor)   # (2, max_length)

    for i in range(2):
        pred = decode_captions(pred_captions[i:i+1], idx_to_word)[0]
        true = decode_captions(sample_captions_true[i:i+1], idx_to_word)[0]
        print(f"Image {i} prediction: {pred}")
        print(f"      GT: {true}\n")

/tmp/ipykernel_3515971/365017052.py:6: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(os.path.join(model_save_root, 'Transformer.pth'), map_location=D

Image 0 prediction: a one way sign <UNK> left on a city street <END>
      GT: <START> a large <UNK> is taking off from the airport <END>

Image 1 prediction: a living room with a <UNK> floor lamp sofa wooden coffee table and end table <END>
      GT: <START> a one way sign <UNK> left on a city street <END>

Image 0 prediction: a sandwich with tomatoes and <UNK> cheese sits next to chips <END>
      GT: <START> a bicycle <UNK> with a clock as the front <UNK> <END>

Image 1 prediction: a room <END>
      GT: <START> a black <UNK> motorcycle parked in front of a <UNK> <END>

